# 02 Skill Extraction

This notebook converts the cleaned job-posting dataset into a normalized job-skills table.

The cleaned jobs dataset contains a `skills_text` column where multiple skills are stored in one cell. This notebook splits that field so each job-skill relationship becomes its own row.

Input:

- `data/processed/cleaned_jobs.csv`

Output:

- `data/processed/job_skills.csv`

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
CLEANED_JOBS_PATH = Path("../data/processed/cleaned_jobs.csv")
JOB_SKILLS_OUTPUT_PATH = Path("../data/processed/job_skills.csv")

In [3]:
jobs_df = pd.read_csv(CLEANED_JOBS_PATH)

jobs_df.head()

,job_id,collection_date,country,city,job_title,company,source,job_url,employment_type,work_mode,...,years_experience_min,years_experience_max,education_requirement,description_short,skills_text,notes,salary_period,has_salary,experience_category,entry_level_with_high_experience
0,UAE_001,2026-05-20,UAE,Dubai,Product Data Analyst Intern,TikTok,GulfTalent,https://www.gulftalent.com/uae/jobs/product-da...,Full-time,NaN,...,NaN,NaN,NaN,Internship analyzing product performance trend...,"SQL, Excel, Tableau, Data Visualization, Dashb...",Internship requires at least 3-month commitment.,NaN,False,Not specified,False
1,UAE_002,2026-05-20,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,Bayt,https://www.bayt.com/en/uae/jobs/project-data-...,NaN,NaN,...,NaN,NaN,NaN,Internship supporting ERP implementation throu...,"Data Cleaning, Quality Assurance, Manual Testi...",Bayt notes the post was translated by AI.,NaN,False,Not specified,False
2,UAE_003,2026-05-20,UAE,Dubai,Data Analyst and Visual Content Designer (UAE ...,Genius HRTech Services,Bayt,https://www.bayt.com/en/uae/jobs/data-analyst-...,Full-time,NaN,...,1.0,3.0,Bachelor's degree / higher diploma,"Entry-level role analyzing HSE datasets, build...","Excel, Power BI, Dashboarding, Reporting, Data...",UAE nationals only.,NaN,True,1 year,False
3,UAE_004,2026-05-20,UAE,Dubai,Associate Data Analyst- UAE Nationals only,Delivery Hero SE,Bayt,https://www.bayt.com/en/uae/jobs/associate-dat...,NaN,NaN,...,2.0,3.0,"Bachelor's degree in Data Science, Statistics,...",Early-career analytics role supporting product...,"SQL, Excel, Tableau, Power BI, Looker, Python,...",UAE nationals only; early-career wording but r...,NaN,False,2 years,False
4,UAE_005,2026-05-20,UAE,Dubai,Associate Data Analyst,Bayut | dubizzle,GulfTalent,https://www.gulftalent.com/uae/jobs/associate-...,Full-time,NaN,...,2.0,3.0,NaN,"Associate strategy analytics role using SQL, E...","SQL, Excel, Dashboarding, Reporting, Data Visu...",NaN,NaN,False,2 years,False


In [4]:
jobs_df.shape

(20, 24)

In [5]:
required_columns = [
    "job_id",
    "country",
    "city",
    "job_title",
    "company",
    "skills_text",
]

missing_columns = [column for column in required_columns if column not in jobs_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [11]:
skills_df = (
    jobs_df[["job_id", "country", "city", "job_title", "company", "skills_text"]]
    .dropna(subset=["skills_text"])
    .assign(skill=lambda x: x["skills_text"].str.split(","))
    .explode("skill")
    .reset_index(drop=True)
)

skills_df["skill"] = skills_df["skill"].astype("string").str.strip()

skills_df = skills_df[
    skills_df["skill"].notna() &
    (skills_df["skill"] != "")
]

skills_df.head(10)

,job_id,country,city,job_title,company,skills_text,skill
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",SQL
1,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Excel
2,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Tableau
3,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Data Visualization
4,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Dashboarding
5,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Reporting
6,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Stakeholder Communication
7,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Presentation Skills
8,UAE_002,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,"Data Cleaning, Quality Assurance, Manual Testi...",Data Cleaning
9,UAE_002,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,"Data Cleaning, Quality Assurance, Manual Testi...",Quality Assurance


In [7]:
skills_df.shape

(145, 7)

In [8]:
skills_df["skill"].value_counts()

skill
Stakeholder Communication    18
Reporting                    17
SQL                          11
Data Visualization           10
Dashboarding                 10
Technical Documentation      10
Excel                         9
Power BI                      9
Python                        9
Presentation Skills           7
Data Cleaning                 7
Tableau                       5
Quality Assurance             5
Manual Testing                4
Statistics                    3
Looker                        2
R                             2
Machine Learning              2
Bug Reporting                 1
AWS                           1
Google Cloud                  1
Azure                         1
ETL                           1
Name: count, dtype: Int64

In [9]:
skills_df[skills_df["skill"].str.contains(
    "Posting|Title|LinkedIn|internship",
    case=False,
    na=False
)]

,job_id,country,city,job_title,company,skills_text,skill


In [10]:
skills_df.to_csv(JOB_SKILLS_OUTPUT_PATH, index=False)

print(f"Saved job skills table to: {JOB_SKILLS_OUTPUT_PATH}")
print(f"Skill rows saved: {len(skills_df)}")

Saved job skills table to: ..\data\processed\job_skills.csv
Skill rows saved: 145


## Skill Extraction Summary

The cleaned job-posting dataset was transformed into a normalized job-skills table. Each row now represents one job-skill relationship.

This structure is more useful than keeping all skills in one comma-separated field because it supports:

- counting how many jobs mention each skill
- comparing skills by country
- joining jobs and skills in SQL
- building Power BI visuals using individual skill values

The output file `job_skills.csv` is used in the market analysis notebook, SQL database, and Power BI dashboard.